[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_4_GIS_analytics/assignments/point_cloud_analysis.ipynb)

# Assignment 4B: Processing lidar and UAS SfM point clouds

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this notebook we analyze an airborne lidar point cloud and a UAS structure from motion (SfM) point cloud of the Mid Pines area at Lake Wheeler and turn them into surfaces. We compare their point densities, bin the lidar returns into raster surfaces and a simple land cover classification, interpolate a bare earth DEM and a canopy DSM with their topographic derivatives, and check the lidar DEM against the surveyed GCPs. The optional last part classifies ground points, builds a voxel model of the canopy, and cuts a transect through the cloud.

The two point clouds represent the same ground in two different ways. Lidar records several returns per pulse and reaches the ground under the canopy; SfM reconstructs the first surface the camera sees. Keep that difference in mind when you compare densities and surfaces.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_4_GIS_analytics/assignments/assignment_4b.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed. The 3D views described on the assignment page are done in the GRASS GUI; the notebook shows the same surfaces in 2D.

## Learning objectives

- Read the point classes and the extent of a LAS file
- Compare lidar and SfM point densities by binning the clouds at the same resolution
- Build binned surfaces from returns and classes, and combine them into a simple classification
- Interpolate a DEM and a DSM with slope and curvature using the regularized spline with tension
- Validate a DEM against surveyed points and report bias and RMSE

## Outline

- Part 1: Environment setup
- Part 2: Explore the point clouds
- Part 3: Point density
- Part 4: Binned surfaces and a simple classification
- Part 5: High resolution DEM and DSM
- Part 6: Validate the lidar DEM against the GCPs
- Part 7 (optional): Ground classification, voxels, and a transect
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab). The conda-forge build includes PDAL, which the point cloud import tools need.

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed, that it has the PDAL import tools, and which Python version is running.

In [ ]:
!grass --version
!grass --tmp-project XY --exec r.in.pdal --help 2>&1 | head -3

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5).

In [ ]:
import json
import math
import os
import subprocess
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB)
- The 2013 lidar point cloud of Mid Pines, classified (LAS, about 165 MB)
- The 2015 UAS SfM sample point cloud of a Mid Pines sub-area (LAS, about 60 MB)
- The surveyed GCPs as a GRASS vector pack
- The fallback DEM and DSM packs, used only if the PDAL import is not available

The downloads total about 500 MB and take a minute or two on Colab.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
LIDAR_URL = f"{BUCKET}/lidar-data/Midpines/mid_pines_spm_2013.las"
UAS_URL = f"{BUCKET}/lidar-data/Midpines/2015_06_sample_points_NCspm.las"
GCP_URL = f"{BUCKET}/grassdata/GCP_12_decimal.pack"
FALLBACK_URL = f"{BUCKET}/lidar-data/Midpines/secref_mid_pines_spm_elev.zip"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o mid_pines_spm_2013.las {LIDAR_URL}
!curl -sSL -o 2015_06_sample_points_NCspm.las {UAS_URL}
!curl -sSL -o GCP_12_decimal.pack {GCP_URL}
!curl -sSL -o secref_mid_pines_spm_elev.zip {FALLBACK_URL} && unzip -q -o secref_mid_pines_spm_elev.zip
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `point_clouds` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.r_in_pdal(...)` runs `r.in.pdal`, `tools.g_region(...)` runs `g.region`, and so on; a flag like `-o` is passed as `flags="o"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "point_clouds"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session)

print(tools.g_proj(flags="p").text)

### A helper

`describe` prints the statistics of a raster in the current region from `r.univar`; we use it on every raster we make.

In [ ]:
def describe(raster, label=None):
    """Print the range, mean, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells")
    print(f"  min {s['min']:8.2f}  max {s['max']:8.2f}  mean {s['mean']:8.2f}  median {s['median']:8.2f}  stddev {s['stddev']:6.2f}")
    return s

---
## Part 2: Explore the point clouds

### Point classes

PDAL ships with the conda GRASS as the `pdal` command. Its `info` tool reads the file's metadata and, with the stats filter, counts the points in each class. The 2013 data uses four codes: 1 unclassified (vegetation and buildings), 2 ground, 7 low point (noise), and 11, reserved in LAS 1.4, which this provider used for withheld points. Classes 1 and 2 are the ones we work with.

In [ ]:
info = json.loads(subprocess.check_output(
    ["pdal", "info", "--stats", "--filters.stats.count=Classification", "mid_pines_spm_2013.las"], text=True
))
for stat in info["stats"]["statistic"]:
    if stat["name"] == "Classification":
        print("Class  Points")
        for item in stat["counts"]:
            value, count = item.split("/")
            print(f"{int(float(value)):5d}  {int(count):>10,}")
print(f"Total points: {info['stats']['statistic'][0]['count']:,}")

### Extent

`r.in.pdal -g` prints the extent of a point cloud without importing it. We set the computational region to the lidar extent; later cells change the resolution and the extent as needed.

In [ ]:
print("Lidar extent:", tools.r_in_pdal(input="mid_pines_spm_2013.las", flags="g").text)
print("UAS extent:  ", tools.r_in_pdal(input="2015_06_sample_points_NCspm.las", flags="g").text)

tools.g_region(n=220218, s=218694, e=637795, w=636271)
print(tools.g_region(flags="p").text)

> **If the PDAL import is not available** (for example, your GRASS build has no PDAL support), restore the interpolated DEM and DSM from the fallback zip under the names Part 5 uses, then skip the import cells and continue with the analysis and the maps:
>
> ```python
> tools.r_unpack(input="mid_pines_lidar2013_dem.pack", output="mid_pines_ground_elev", flags="o", overwrite=True)
> tools.r_unpack(input="mid_pines_lidar2013_dsm.pack", output="mid_pines_surface_elev", flags="o", overwrite=True)
> ```
>
> The zip holds four packs; the two `secref_*` files cover a different site and are not used.

---
## Part 3: Point density

### Lidar: all points and ground points

Set the resolution to 1 m; the `-a` flag aligns the region boundaries to whole multiples of the resolution. Then count all points per cell with binning (`method=n`), and the ground points only (`class_filter=2`). The `-o` flag skips the projection check: the file's CRS record is worded differently from the project's, but it is the same coordinate system. A few thousand points that fall on the region boundary are skipped, so the processed count is slightly lower than the file's point count.

In [ ]:
tools.g_region(res=1, flags="a")

tools.r_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_dens_all", method="n", flags="o", overwrite=True)
tools.r_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_dens_ground", class_filter="2", method="n", flags="o", overwrite=True)

_ = describe("mid_pines_dens_all", "Lidar, all returns per 1 m cell")
_ = describe("mid_pines_dens_ground", "Lidar, ground points per 1 m cell")

### UAS: the sub-area

The UAS sample covers a sub-area of the lidar. Set a 1 m region over it and bin it the same way. Then give all three density rasters the same histogram equalized color table (`r.colors -e`) so they can be compared.

In [ ]:
tools.g_region(n=219660, s=219288, e=637287, w=636876, res=1, flags="a")
tools.r_in_pdal(input="2015_06_sample_points_NCspm.las", output="midpines_uas_06_1m_dens", method="n", flags="o", overwrite=True)
_ = describe("midpines_uas_06_1m_dens", "UAS SfM points per 1 m cell")

_ = tools.r_colors(map="mid_pines_dens_all,mid_pines_dens_ground,midpines_uas_06_1m_dens", color="bcyr", flags="e")

#### Map: Lidar all-return, lidar ground, and UAS densities over the UAS sub-area

In [ ]:
for raster, title in [
    ("mid_pines_dens_all", "Lidar, all returns"),
    ("mid_pines_dens_ground", "Lidar, ground"),
    ("midpines_uas_06_1m_dens", "UAS SfM"),
]:
    m = gj.Map(width=500, height=460, use_region=True)  # zoom to the UAS sub-area
    m.d_rast(map=raster)
    m.d_legend(raster=raster, at=(5, 50, 3, 7), title=title, flags="sb", bgcolor="white")
    m.d_barscale(at=(55, 5))
    m.show()

> **Think about it:** Where does the lidar ground density drop to zero, and why? Where is the UAS density highest? Lidar records several returns per pulse and reaches the ground under the canopy; SfM reconstructs the first surface the camera sees and finds the most points where the photos have the most texture.

---
## Part 4: Binned surfaces and a simple classification

Binning assigns each cell a statistic of the points that fall in it, with no interpolation. Different returns and classes give different surfaces. We set a 2 m region over the whole lidar extent and build three of them:

- the highest point in each cell from classes 1 and 2, which is a DSM
- the mean of the last returns, which in `r.in.pdal` keeps the last return only where the pulse produced more than one return, so this surface marks the canopy, not the ground
- the mean of the points already classified as ground

In [ ]:
tools.g_region(n=220218, s=218694, e=637795, w=636271, res=2, flags="a")

tools.r_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_all_max", method="max", class_filter="1,2", flags="o", overwrite=True)
tools.r_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_last_mean", method="mean", return_filter="last", class_filter="1,2", flags="o", overwrite=True)
tools.r_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_ground_mean", class_filter="2", flags="o", overwrite=True)

for raster in ("mid_pines_all_max", "mid_pines_last_mean", "mid_pines_ground_mean"):
    _ = describe(raster)

### Classes from the three surfaces

A cell with a multi-return last echo is vegetation (2); a cell with a ground point but no multi-return echo is ground (1); a cell with a first return and neither is a building (3). `r.category` labels the classes and `r.report` sums their area.

In [ ]:
tools.r_mapcalc(
    expression="classes = if(not(isnull(mid_pines_last_mean)), 2, if(not(isnull(mid_pines_ground_mean)), 1, if(not(isnull(mid_pines_all_max)), 3, null())))",
    overwrite=True,
)
gs.write_command("r.category", map="classes", rules="-", separator=":", stdin="1:ground\n2:vegetation\n3:buildings")
gs.write_command("r.colors", map="classes", rules="-", stdin="1 220:220:180\n2 0:180:0\n3 150:0:0", quiet=True)

print(tools.r_report(map="classes", units="h,p", flags="n").text)

#### Map: Ground, vegetation, and buildings from the binning rule

In [ ]:
classes_map = gj.Map(width=600, height=600)
classes_map.d_rast(map="classes")
classes_map.d_legend(raster="classes", at=(5, 20, 2, 6), flags="cb", bgcolor="white", title="Class")
classes_map.d_barscale(at=(55, 5))
classes_map.show()

> **Think about it:** Zoom into the map (set a smaller region with `tools.g_region` and `use_region=True`) and look for places where the rule fails: tree crowns with no multiple returns, buildings under trees, cars and sheds. What does each failure say about the rule?

---
## Part 5: High resolution DEM and DSM

### Import the points as vectors

`v.in.pdal` imports the points as a vector map: the ground class for the DEM and the first returns of classes 1 and 2 for the DSM. `v.outlier -e` estimates the point density and mean spacing of the ground points, which tells us what resolution the interpolation can support.

In [ ]:
tools.v_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_ground", class_filter="2", flags="o", overwrite=True)
tools.v_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_surface", class_filter="1,2", return_filter="first", flags="o", overwrite=True)

print(tools.v_outlier(input="mid_pines_ground", flags="e").text)

### Interpolate with the regularized spline with tension

We set a smaller region at 0.5 m resolution and interpolate both surfaces with `v.surf.rst`, computing slope and profile curvature at the same time. Each surface takes under a minute. A higher `npmin=150` reduces the segmentation artifacts visible in the slope and curvature maps, at a much longer run time.

In [ ]:
tools.g_region(n=219780, s=219100, e=637250, w=636575, res=0.5, flags="a")

for points, name in [("mid_pines_ground", "mid_pines_ground"), ("mid_pines_surface", "mid_pines_surface")]:
    tools.v_surf_rst(
        input=points,
        elevation=f"{name}_elev",
        slope=f"{name}_slope",
        pcurv=f"{name}_profcurv",
        npmin=80,
        tension=20,
        smooth=1,
        overwrite=True,
    )
tools.r_colors(map="mid_pines_ground_elev,mid_pines_surface_elev", color="elevation")

_ = describe("mid_pines_ground_elev", "DEM (ground)")
_ = describe("mid_pines_surface_elev", "DSM (first returns)")

#### Map: DEM and DSM

In [ ]:
for raster, title in [("mid_pines_ground_elev", "DEM (m)"), ("mid_pines_surface_elev", "DSM (m)")]:
    m = gj.Map(width=500, height=500, use_region=True)
    m.d_rast(map=raster)
    m.d_legend(raster=raster, at=(5, 40, 2, 6), title=title)
    m.d_barscale(at=(55, 5))
    m.show()

#### Map: Slope of the DEM and of the DSM

In [ ]:
for raster, title in [("mid_pines_ground_slope", "DEM slope (deg)"), ("mid_pines_surface_slope", "DSM slope (deg)")]:
    m = gj.Map(width=500, height=500, use_region=True)
    m.d_rast(map=raster)
    m.d_legend(raster=raster, at=(5, 40, 2, 6), title=title)
    m.d_barscale(at=(55, 5))
    m.show()

#### Map: The DSM colored by the classes

In the GUI this is the 3D view with the `classes` raster as the color; here the DSM is shown as shaded relief with the classes on top.

In [ ]:
tools.r_relief(input="mid_pines_surface_elev", output="mid_pines_surface_relief", overwrite=True)

shade_map = gj.Map(width=600, height=600, use_region=True)
shade_map.d_shade(shade="mid_pines_surface_relief", color="classes")
shade_map.d_legend(raster="classes", at=(5, 20, 2, 6), flags="cb", bgcolor="white", title="Class")
shade_map.d_barscale(at=(55, 5))
shade_map.show()

> **Think about it:** What do the slope and curvature maps show that the elevation does not? Look for the field terraces, the road, the segmentation seams from `npmin`, and the difference between the DEM and the DSM at the tree lines.

---
## Part 6: Validate the lidar DEM against the GCPs

Restore the surveyed GCPs and add columns for the DEM height and the difference. The pack was written by an older GRASS whose CRS record differs in wording from the current project's; `-o` accepts the project's definition. `v.what.rast -i` samples the DEM at each GCP, interpolating between the four nearest cells.

In [ ]:
tools.v_unpack(input="GCP_12_decimal.pack", flags="o", overwrite=True)
tools.v_db_addcolumn(map="GCP_12_decimal", columns="dem_height DOUBLE, height_difference DOUBLE")
tools.v_what_rast(map="GCP_12_decimal", raster="mid_pines_ground_elev", column="dem_height", flags="i")
tools.v_db_update(map="GCP_12_decimal", column="height_difference", query_column="ASL - dem_height")

rows = tools.v_db_select(map="GCP_12_decimal", columns="GCP,ASL,dem_height,height_difference", format="json").json["records"]
print(f"{'GCP':8s} {'Survey (m)':>11s} {'DEM (m)':>9s} {'Diff (m)':>9s}")
for r in rows:
    print(f"{r['GCP']:8s} {r['ASL']:11.3f} {r['dem_height']:9.3f} {r['height_difference']:9.3f}")

### Bias and accuracy

Read the numbers the way Topic 3B read checkpoint residuals: the mean of the differences is the bias of the lidar DEM against the survey, and the RMSE is its vertical accuracy at these twelve points. The standard deviation is the spread around the bias; RMSE combines the two.

In [ ]:
diffs = [r["height_difference"] for r in rows]
n = len(diffs)
mean = sum(diffs) / n
stddev = math.sqrt(sum((d - mean) ** 2 for d in diffs) / n)
rmse = math.sqrt(sum(d * d for d in diffs) / n)

print(f"n = {n}")
print(f"mean (bias):        {mean:+.3f} m  (survey minus DEM; negative means the DEM is above the survey)")
print(f"standard deviation: {stddev:.3f} m")
print(f"RMSE:               {rmse:.3f} m")
print(f"min, max:           {min(diffs):+.3f}, {max(diffs):+.3f} m")

#### Map: The GCPs colored by their height difference over the DEM

In [ ]:
tools.g_region(n=219780, s=219100, e=637250, w=636575, res=0.5, flags="a")
gcp_map = gj.Map(width=600, height=600, use_region=True)
gcp_map.d_rast(map="mid_pines_ground_elev")
gcp_map.d_vect_thematic(
    map="GCP_12_decimal",
    column="height_difference",
    algorithm="int",
    nclasses=4,
    colors="blue,30:144:255,173:216:230,255:192:203",
    icon="basic/circle",
    size=14,
)
gcp_map.d_vect(map="GCP_12_decimal", attribute_column="GCP", label_color="black", label_size=10, color="none", fill_color="none", flags="s")
gcp_map.d_legend_vect(at=(2, 30), flags="b", title="Survey minus DEM (m)")
gcp_map.d_barscale(at=(55, 5))
gcp_map.show()

> **Think about it:** Is the bias within the vertical accuracy you would expect from the 2013 county lidar? Which GCPs differ most, and what is at those locations (field edge, slope, vegetation)? Remember that the survey and the lidar are two years apart.

---
## Part 7 (optional): Ground classification, voxels, and a transect

### Ground classification with `v.lidar.mcc`

The [v.lidar.mcc](https://grass.osgeo.org/grass-stable/manuals/addons/v.lidar.mcc.html) addon separates ground from non-ground points with a multiscale curvature classification. We install it with `g.extension`, import a small unclassified sample (`-r` limits the import to the region), and classify it. Addons installed after the `Tools` object was created are called with `grass.script`.

In [ ]:
tools.g_extension(extension="v.lidar.mcc")

tools.g_region(n=219511, s=219354, w=637071, e=637249, res=1)
tools.v_in_pdal(input="mid_pines_spm_2013.las", output="mid_pines_sample", flags="ro", overwrite=True)
gs.run_command("v.lidar.mcc", input="mid_pines_sample", ground="ground", nonground="nonground", overwrite=True)

for name in ("mid_pines_sample", "ground", "nonground"):
    print(f"{name}: {tools.v_info(map=name, flags='t').keyval['points']:,} points")

#### Map: Ground (brown) and non-ground (green) points from MCC

In [ ]:
mcc_map = gj.Map(width=600, height=530, use_region=True)
mcc_map.d_vect(map="ground", color="160:110:60", icon="basic/point", size=3)
mcc_map.d_vect(map="nonground", color="0:140:0", icon="basic/point", size=3)
mcc_map.d_barscale(at=(55, 5))
mcc_map.show()

Compare the result with the provider's class 2: `v.in.pdal` with `class_filter=2` over the same region gives the provider's ground points.

### Point density in 3D

Set a small region with a vertical extent (`b`, `t`, `tbres`) for a 3D raster. GRASS has no PDAL based 3D raster import, so we export the points inside this region to a text file with PDAL (about 100,000 points) and bin the text file with [r3.in.xyz](https://grass.osgeo.org/grass-stable/manuals/r3.in.xyz.html). `r3.in.xyz` reads the file once per vertical slice, which is why the crop matters.

In [ ]:
tools.g_region(n=219502, s=219348, w=637070, e=637276, b=110, t=135, res=2, res3=2, tbres=0.5, flags="3")

subprocess.run([
    "pdal", "translate", "mid_pines_spm_2013.las", "mid_pines_sub.txt", "crop",
    "--filters.crop.bounds=([637070,637276],[219348,219502])",
    "--writers.text.order=X,Y,Z", "--writers.text.keep_unspecified=false", "--writers.text.write_header=false",
], check=True)
tools.r3_in_xyz(input="mid_pines_sub.txt", output="mid_pines_dens", method="n", separator="comma", overwrite=True)
gs.write_command("r3.colors", map="mid_pines_dens", rules="-", stdin="0% 255:255:100\n5% green\n100% red", quiet=True)

print(tools.r3_info(map="mid_pines_dens", flags="g").text)

#### Map: Horizontal slices through the canopy

The GUI shows the 3D raster with isosurfaces and slices in the 3D view. Here `r3.to.rast` splits it into one 2D raster per vertical level (50 slices from 110 to 135 m), and we draw three of them: near the ground, in the lower canopy, and in the upper canopy.

In [ ]:
tools.r3_to_rast(input="mid_pines_dens", output="dens_slice", overwrite=True)

for level, height in [(5, 112.5), (21, 120.5), (37, 128.5)]:
    name = f"dens_slice_{level:05d}"
    tools.r_colors(map=name, raster_3d="mid_pines_dens")
    m = gj.Map(width=420, height=330, use_region=True)
    m.d_rast(map=name)
    m.d_legend(raster=name, at=(5, 40, 2, 6), title=f"Points at {height} m")
    m.show()

### Lidar transect

The [v.profile.points](https://grass.osgeo.org/grass-stable/manuals/addons/v.profile.points.html) addon cuts a transect of a chosen width through the point cloud, straight from the LAS file, and rotates it into a vertical plane. The output's x axis is the distance along the transect and its y axis is the elevation, so the map below is a side view. The output's z coordinate is the offset of each point from the transect line, not its elevation, so the points are drawn in one color.

In [ ]:
tools.g_extension(extension="v.profile.points")
tools.g_region(n=219511, s=219354, w=637071, e=637249, res=1)
gs.run_command(
    "v.profile.points",
    file_input="mid_pines_spm_2013.las",
    output="profile",
    width=5,
    coordinates=[637106, 219373, 637132, 219451],
    overwrite=True,
)
# The transect is in its own coordinates (distance, elevation); set the region to it.
tools.g_region(vector="profile")
profile_map = gj.Map(width=800, height=300, use_region=True)
profile_map.d_vect(map="profile", color="30:60:90", icon="basic/point", size=3)
profile_map.show()

> **Think about it:** The transect crosses a tree line. Where are the ground returns under the crowns, and how far into the canopy do the points reach?

---
## Report questions

Answer each question in the Markdown cell below it. These answers, with the maps from this notebook, are the content of your report (see the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_4_GIS_analytics/assignments/assignment_4b.html) for the submission details).

#### Question 1: Point density

Compare the lidar all-point and ground-point densities and the UAS density over the sub-area, shown with the same color table. Where and why do they differ?

`Add text here with your answer to question 1`

#### Question 2: Classification

Describe the `classes` raster, with examples of where the binning rule identifies buildings correctly and where it fails, and why.

`Add text here with your answer to question 2`

#### Question 3: Surfaces

Describe the interpolated DEM and DSM with their slope or curvature maps and the 3D view from the GUI. What do the derivatives show that the elevation does not?

`Add text here with your answer to question 3`

#### Question 4: Validation

Give the GCP table with the DEM height and the difference, its mean and RMSE, and what they say about the vertical accuracy and the bias of the lidar DEM.

`Add text here with your answer to question 4`

#### Question 5: Optional parts

Figures and a short interpretation for any of Part 7 you completed.

`Add text here with your answer to question 5`